# HW2 Problem 1 · A Taxi Neural Network for a 12-Year-Old Learning It for the First Time

**Fully runnable version**: the code below chains together every statement from the original `.py` file as-is; only the input file path has been fixed to `taxi_trips.parquet` so that it runs in Jupyter. Run it one cell at a time while reading the explanatory comments after each `#`.

Required packages: `numpy`, `pandas`, `torch`, `pyarrow`. The data file must be in the same folder as the notebook (in the repository, copy `data/taxi_trips.parquet` into this folder). The same code also runs from the `homework_2` folder with `python run_all.py --problem 1`.

## Warm-up — getting a feel for uncertainty as numbers

Two taxi routes both have an average of 20 minutes. But if route A takes [19,20,21] minutes and route B takes [10,20,30] minutes, they differ in how stable they are. The mean is the **baseline expected travel time**, and the standard deviation is **how much it wobbles**.

- `μ(x)` is the expected time for input x.
- `σ(x)` is how much the expected time wobbles, and it must always be positive.
- If `g(x)` outputs two real numbers, the first one is μ, and the second is turned into a positive number with `softplus` to make σ.

Negative log-likelihood of the normal model:

$$\ell_i=\tfrac12\log(2\pi)+\log\sigma_i+\tfrac{(y_i-\mu_i)^2}{2\sigma_i^2}$$

The second term of this formula keeps the model from abusing a wide standard deviation, and the last term keeps it from being overconfident about its errors with a narrow standard deviation.

In [ ]:
from math import erf, sqrt, log, pi
mean = 40.0  # mean travel time of 40 minutes
for sd in [5.0, 10.0]:  # even with the same mean, if the uncertainty differs...
    z = (45.0 - mean) / sd  # convert the 45-minute threshold into standard-deviation units
    probability = 0.5 * (1.0 + erf(z / sqrt(2.0)))  # standard normal CDF Φ(z)
    print(f"mean={mean}, std={sd}: P(Y<45)={probability:.4f}")


## What is the Brier score?

If the actual trip took 43 minutes, the event `Y<45` is true, so `z=1`; if it took 48 minutes, it is false, so `z=0`. For a predicted probability p, the Brier loss is `(p-z)^2`. With `p=.8413`, an actual 43-minute trip gives `(.8413-1)^2≈.0252`, and a 48-minute trip gives `.8413^2≈.7078`.

Accuracy binarizes the prediction by checking whether p≥0.5, so it treats probabilities of 0.51 and 0.99 about the same. If about 99.55% of the data are under 45 minutes, accuracy alone makes it hard to tell how well the model detects the difficult (long) trips.

## Code 1. 1. Imports and numerical-computing packages (nn, softplus, numpy, pandas)

Before running this block, make sure you have run all previous code cells from top to bottom. No dependencies on functions, variables, the model, or the data have been omitted.

In [ ]:
import sys  # reads the data file name passed in from the terminal.
import math  # uses the normal-distribution constant pi and the logarithm.
import numpy as np  # creates a reproducible data split.
import pandas as pd  # reads the taxi table with named columns.
import torch  # handles array computation and automatic differentiation.
from torch import nn  # imports the neural-network layers.
from torch.nn.functional import softplus  # turns a real number into a positive standard deviation.



## Code 2. 2. Experiment seed · reading the data · Train/Validation/Test split and model definition

Before running this block, make sure you have run all previous code cells from top to bottom. No dependencies on functions, variables, the model, or the data have been omitted.

In [ ]:
torch.manual_seed(7)  # reproduces the initial weights and batch order under the same run conditions.
torch.set_num_threads(2)  # uses two threads for CPU computation.
df = pd.read_parquet("taxi_trips.parquet")  # opens the table where each row is one trip.
y = torch.tensor(df.pop("trip_duration").to_numpy(), dtype=torch.float32)  # separates out the target to prevent leakage into the inputs.
X = torch.tensor(df.to_numpy(), dtype=torch.float32)  # turns the remaining seven inputs into an N-row, 7-column tensor.
idx = np.random.default_rng(42).permutation(len(y))  # shuffles all row indices into a random order.
tr, va, te = np.split(idx, [int(.70*len(y)), int(.85*len(y))])  # assigns disjoint rows to training, validation, and test.
net = nn.Sequential(nn.Linear(7,64), nn.ReLU(), nn.Linear(64,2))  # for each trip, computes two numbers via 64 hidden features.
opt = torch.optim.Adam(net.parameters(), lr=.002)  # creates Adam, which adapts the update size for each weight.


## Code 3. 3. Mean/standard-deviation transformation and the Normal NLL function

Before running this block, make sure you have run all previous code cells from top to bottom. No dependencies on functions, variables, the model, or the data have been omitted.

In [ ]:
def predict(x):  # returns the conditional mean and standard deviation for multiple trips.
    raw = net(x)  # gets a B-row, 2-column output for B trips.
    return raw[:,0], softplus(raw[:,1]) + .001  # the first column is the center in minutes; the second is the width made positive.
def nll(yb, mu, sd):  # computes how well the normal distribution explains the actual times.
    return (.5*math.log(2*math.pi)+sd.log()+.5*((yb-mu)/sd)**2).mean()  # averages the width cost and the standardized-residual cost.


## Code 4. 4. Training loop · AutoGrad · choosing the epoch with Validation

Before running this block, make sure you have run all previous code cells from top to bottom. No dependencies on functions, variables, the model, or the data have been omitted.

In [ ]:
best, wait = math.inf, 0  # at first, any finite validation score counts as an improvement; wait counts epochs without improvement.
for epoch in range(300):  # passes over the training data at most 300 times; stops early once validation NLL stops improving.
    net.train()  # puts the model in training mode.
    order = torch.as_tensor(tr)[torch.randperm(len(tr))]  # reshuffles only the training rows.
    for batch in order.split(2048):  # processes every batch, including the last smaller one.
        opt.zero_grad()  # clears the gradients from the previous batch.
        loss = nll(y[batch], *predict(X[batch]))  # the asterisk passes the mean and standard deviation as separate arguments.
        loss.backward()  # differentiates the loss with respect to every weight.
        opt.step()  # updates the weights once using the computed gradients.
    net.eval()  # switches to evaluation mode.
    with torch.no_grad():  # no need to record gradients during validation.
        score = nll(y[va], *predict(X[va])).item()  # gets the validation loss as a Python number.
    print("epoch", epoch+1, "validation NLL", round(score, 6))  # records the validation curve in the log.
    if score < best:  # checks whether the validation NLL decreased.
        best, chosen_epoch = score, epoch+1  # records the best score and its epoch.
        state = {k:v.detach().clone() for k,v in net.state_dict().items()}  # makes a copy of the weights independent of later updates.
        wait = 0  # improved, so reset the wait counter.
    else:  # validation NLL did not decrease this epoch.
        wait += 1  # counts one more epoch without improvement.
        if wait >= 30:  # 30 epochs in a row without improvement: further training is not helping.
            break  # early stopping.
print("epochs run", epoch+1)  # prints how many epochs were actually trained.


## Code 5. 5. Restoring the best weights

Before running this block, make sure you have run all previous code cells from top to bottom. No dependencies on functions, variables, the model, or the data have been omitted.

In [ ]:
net.load_state_dict(state)  # restores the model selected by validation.


## Code 6. 6. Test probabilities · Brier · Accuracy evaluation

Before running this block, make sure you have run all previous code cells from top to bottom. No dependencies on functions, variables, the model, or the data have been omitted.

In [ ]:
def report(name, mu, sd):  # evaluates each model on the same test rows.
    z = y[te] < 45  # true if under 45 minutes; exactly 45 minutes is false.
    p = torch.special.ndtr(((45-mu)/sd).double())  # computes the area to the left of the standardized threshold.
    decision = p >= .5  # applies the probability threshold for predicting the event.
    brier = ((p-z.double())**2).mean()  # averages the squared distance between the probability and the actual 0 or 1.
    acc = (decision == z).double().mean()  # computes the fraction of trips predicted correctly.
    late_hits = ((~decision) & (~z)).sum()  # number of trips that actually took 45 minutes or more and that the model also judged as such.
    print(name, float(nll(y[te],mu,sd)), brier.item(), acc.item(), late_hits.item())  # prints the evaluation results for each model.
with torch.no_grad():  # the final test is performed after training is finished.
    report("constant", y[tr].mean(), y[tr].std(unbiased=False))  # baseline model using the training mean and the population-formula standard deviation.
    report("MLP", *predict(X[te]))  # uses a different mean and width for each trip.
print("epoch, validation NLL", chosen_epoch, best)  # prints the basis for the model selection.


## Test results (identical when rerun with `python run_all.py --problem 1`)

| Model | Continuous test NLL | Brier ↓ | Accuracy |
|---|---:|---:|---:|
| Constant Normal | 3.476824 | 0.004495 | 99.5505% |
| Heteroscedastic MLP | 3.280044 | 0.003973 | 99.5505% |

The test split contained 353 taxi trips of 45 minutes or longer, but the MLP detected only 1 of them as positive. The MLP improved the Brier score, but its Accuracy stayed the same. The final answer must explain why these two metrics convey different messages.